In [1]:
!pip install xgboost optuna
import pandas as pd
import numpy as np
import xgboost as xgb
import optuna
from sklearn.model_selection import train_test_split, KFold
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import mean_absolute_error

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 383.6/383.6 kB 9.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 231.8/231.8 kB 13.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.5/78.5 kB 6.8 MB/s eta 0:00:00


In [2]:
#Load datasets
train_df = pd.read_csv("train.csv")
test_df = pd.read_csv("test.csv")
economic_indicators_df = pd.read_csv("EconomicIndicators.csv")

#Fill missing values in InventoryRatio with median
train_df['InventoryRatio'].fillna(train_df['InventoryRatio'].median(), inplace=True)
test_df['InventoryRatio'].fillna(test_df['InventoryRatio'].median(), inplace=True)

#Convert Quarter to numeric
train_df['Quarter'] = train_df['Quarter'].str.extract('(\d+)').astype(int)
test_df['Quarter'] = test_df['Quarter'].str.extract('(\d+)').astype(int)

#Convert Quarter to Month (approximate)
train_df['Month'] = train_df['Quarter'].apply(lambda x: (x - 1) * 3 + 1)
test_df['Month'] = test_df['Quarter'].apply(lambda x: (x - 1) * 3 + 1)

#Merge economic indicators based on Month
train_merged = train_df.merge(economic_indicators_df, on='Month', how='left')
test_merged = test_df.merge(economic_indicators_df, on='Month', how='left')

#Drop Month column
train_merged.drop(columns=['Month'], inplace=True)
test_merged.drop(columns=['Month'], inplace=True)

<ipython-input-2-166beea6d960>:7: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  train_df['InventoryRatio'].fillna(train_df['InventoryRatio'].median(), inplace=True)
<ipython-input-2-166beea6d960>:8: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].meth

In [3]:
#Define categorical features
categorical_features = ['Company', 'Region', 'Industry', 'Bond rating', 'Stock rating']

#Label encode categorical features
label_encoders = {}
for col in categorical_features:
    le = LabelEncoder()
    train_merged[col] = le.fit_transform(train_merged[col])
    test_merged[col] = le.transform(test_merged[col])
    label_encoders[col] = le

In [4]:
#Interaction Features
train_merged["Revenue_Marketshare"] = train_merged["RevenueGrowth"] * train_merged["Marketshare"]
train_merged["Inventory_Quick"] = train_merged["InventoryRatio"] * train_merged["QuickRatio"]

test_merged["Revenue_Marketshare"] = test_merged["RevenueGrowth"] * test_merged["Marketshare"]
test_merged["Inventory_Quick"] = test_merged["InventoryRatio"] * test_merged["QuickRatio"]

#Rolling Averages for Economic Indicators
rolling_features = ["Consumer Sentiment", "Interest Rate", "PMI", "Money Supply",
                    "NationalEAI", "EastEAI", "WestEAI", "SouthEAI", "NorthEAI"]

for feature in rolling_features:
    train_merged[f"{feature}_rolling"] = train_merged[feature].rolling(window=3, min_periods=1).mean()
    test_merged[f"{feature}_rolling"] = test_merged[feature].rolling(window=3, min_periods=1).mean()

In [5]:
#Define features and target (EXCLUDING Prev_Sales)
features_to_exclude = ['Sales', 'Prev_Sales']  # Ensure 'Prev_Sales' is not included
X = train_merged.drop(columns=[col for col in features_to_exclude if col in train_merged.columns])
y = train_merged['Sales']

#Split into train and validation sets
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

In [6]:
def objective(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 500, 1500, step=100),
        'max_depth': trial.suggest_int('max_depth', 4, 12),
        'learning_rate': trial.suggest_loguniform('learning_rate', 0.005, 0.05),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'reg_alpha': trial.suggest_loguniform('reg_alpha', 1e-3, 1.0),
        'reg_lambda': trial.suggest_loguniform('reg_lambda', 1e-3, 1.0),
    }

    model = xgb.XGBRegressor(objective="reg:squarederror", random_state=42, **params)

#5-Fold Cross Validation
    kf = KFold(n_splits=5, shuffle=True, random_state=42)
    mae_scores = []

    for train_index, val_index in kf.split(X_train):
        X_t, X_v = X_train.iloc[train_index], X_train.iloc[val_index]
        y_t, y_v = y_train.iloc[train_index], y_train.iloc[val_index]

        model.fit(X_t, y_t)
        preds = model.predict(X_v)
        mae_scores.append(mean_absolute_error(y_v, preds))

    return np.mean(mae_scores)

#Run Optuna optimization
study = optuna.create_study(direction='minimize')
study.optimize(objective, n_trials=25)

#Get the best parameters
best_params = study.best_params
print("Best XGBoost Parameters:", best_params)

[I 2025-03-19 22:48:31,747] A new study created in memory with name: no-name-110ade98-88c1-405e-9d22-d03e400638e1
<ipython-input-6-7b32c4dc1004>:5: FutureWarning: suggest_loguniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v3.0.0. Use suggest_float(..., log=True) instead.
  'learning_rate': trial.suggest_loguniform('learning_rate', 0.005, 0.05),
<ipython-input-6-7b32c4dc1004>:8: FutureWarning: suggest_loguniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v3.0.0. Use suggest_float(..., log=True) instead.
  'reg_alpha': trial.suggest_loguniform('reg_alpha', 1e-3, 1.0),
<ipython-input-6-7b32c4dc1004>:9: FutureWarning: suggest_loguniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v3.0.0. Use suggest_float(..., log=True) instead.
  'reg_lambda': trial.suggest

Best XGBoost Parameters: {'n_estimators': 1500, 'max_depth': 4, 'learning_rate': 0.01233495830547662, 'subsample': 0.7490706819435389, 'colsample_bytree': 0.8253206410183462, 'reg_alpha': 0.6644228568258758, 'reg_lambda': 0.7956554729855164}


In [7]:
#Train with best parameters
best_xgb_model = xgb.XGBRegressor(objective="reg:squarederror", random_state=42, **best_params)
best_xgb_model.fit(X_train, y_train)

#Predict on validation set
y_val_pred_tuned = best_xgb_model.predict(X_val)

#Evaluate performance
tuned_mae_xgb = mean_absolute_error(y_val, y_val_pred_tuned)
print(f"Tuned XGBoost MAE after Optuna: {tuned_mae_xgb:.2f}")

Tuned XGBoost MAE after Optuna: 1060.17


In [8]:
#Ensure test features match training features
features_to_exclude = ['RowID', 'Prev_Sales']
test_features = test_merged.drop(columns=[col for col in features_to_exclude if col in test_merged.columns])

#Predict sales for test dataset
test_predictions = best_xgb_model.predict(test_features)

#Prepare submission file
submission = pd.DataFrame({'RowID': test_df['RowID'], 'Sales': test_predictions})
submission.to_csv("optimized_xgboost_submission.csv", index=False)

print("Submission file saved as 'optimized_xgboost_submission.csv'.")

Submission file saved as 'optimized_xgboost_submission.csv'.
